In [1]:
import numpy as np
from numba import njit
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigs
import multiprocessing as mp
import time
import os

# --- 1. JIT 加速内核 (固定 k=0) ---
@njit(fastmath=True, nogil=True)
def compute_autonomous_matrix(u_c, steps, n_bins):
    x = 0.5
    # 自治系统直接热启动
    for i in range(500000):
        x = 1 - u_c * x**2
    
    counts = np.zeros((n_bins, n_bins), dtype=np.float64) 
    last_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
    
    for i in range(steps):
        x = 1 - u_c * x**2
        current_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
        if 0 <= current_bin < n_bins:
            counts[last_bin, current_bin] += 1
            last_bin = current_bin
    return counts

# --- 2. 统计评估任务 ---
def evaluate_resolution(n_bins):
    U_C = 1.543689012  
    STEPS = 10**9     # 验证分辨率时，10^9 步已足够
    
    t0 = time.time()
    try:
        counts = compute_autonomous_matrix(U_C, STEPS, n_bins)
        P = csr_matrix(counts)
        row_sums = np.array(P.sum(axis=1)).flatten()
        row_sums[row_sums == 0] = 1.0
        P = P.multiply(1.0 / row_sums[:, np.newaxis])
        
        # 提取 300 个特征值以获得更稳健的间距分布
        vals, _ = eigs(P, k=300, which='LM', tol=1e-7)
        phases = np.sort(np.angle(vals[np.abs(vals) > 0.5]))
        
        # 计算标准差 sigma
        if len(phases) > 10:
            s = np.diff(phases) / np.mean(np.diff(phases))
            sigma = np.std(s)
            
            save_path = f"sigma_res_bins_{n_bins}.npy"
            np.save(save_path, phases)
            
            msg = f"[Res={n_bins}] Sigma={sigma:.4f} | Time: {time.time()-t0:.1f}s"
        else:
            msg = f"[Res={n_bins}] Fail: Too few eigenvalues"
            
    except Exception as e:
        msg = f"[Res={n_bins}] Error: {str(e)}"
    
    print(msg)
    return msg

if __name__ == "__main__":
    # 扫描不同的分辨率，从 12000 到 36000
    resolutions = [12000, 16000, 20000, 24000, 28000, 32000]
    
    print(f">>> 开始分辨率优化扫描 | 算力: 256核/480GB")
    
    # 由于高分辨率矩阵占用内存较大，建议进程数设小一点（如 8-16），防止 OOM
    with mp.Pool(processes=8) as pool:
        pool.map(evaluate_resolution, resolutions)

>>> 开始分辨率优化扫描 | 算力: 256核/480GB
[Res=12000] Sigma=0.7319 | Time: 8.4s
[Res=16000] Sigma=0.7430 | Time: 11.7s
[Res=20000] Sigma=0.6793 | Time: 14.7s
[Res=24000] Sigma=0.7152 | Time: 18.7s
[Res=28000] Sigma=0.7294 | Time: 23.0s
[Res=32000] Sigma=0.7766 | Time: 27.9s
